# Day 2 — Data Cleaning & Database Loading

This notebook demonstrates the complete Day 2 pipeline:
1. Clean NAV history (parse dates, forward-fill weekends/holidays, remove duplicates)
2. Clean investor transactions (standardize types, validate amounts & KYC)
3. Clean scheme performance (coerce numerics, validate ranges)
4. Verify the SQLite database tables loaded by `db_loader.py`
5. Execute the 10 analytical queries from `sql/queries.sql`

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

BASE_DIR       = Path().resolve().parent
RAW_DIR        = BASE_DIR / "data" / "raw"
PROCESSED_DIR  = BASE_DIR / "data" / "processed"
DB_PATH        = BASE_DIR / "data" / "db" / "bluestock_mf.db"
SQL_DIR        = BASE_DIR / "sql"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raw data   : {RAW_DIR}")
print(f"Processed  : {PROCESSED_DIR}")
print(f"Database   : {DB_PATH}")

Raw data   : C:\Users\Dell\Desktop\Internship_coding\data\raw
Processed  : C:\Users\Dell\Desktop\Internship_coding\data\processed
Database   : C:\Users\Dell\Desktop\Internship_coding\data\db\bluestock_mf.db


## 1. Clean NAV History

- Parse dates to datetime
- Sort by `amfi_code` then `date`
- Reindex to full business-day range per fund & forward-fill gaps
- Remove duplicates and validate `nav > 0`

In [2]:
nav_raw = pd.read_csv(RAW_DIR / "02_nav_history.csv")
print(f"Raw NAV records: {len(nav_raw):,}")
print(f"Duplicates     : {nav_raw.duplicated().sum()}")
print(f"Null NAVs      : {nav_raw['nav'].isnull().sum()}")

# Remove duplicates
nav = nav_raw.drop_duplicates()

# Parse dates
nav['date'] = pd.to_datetime(nav['date'], format='mixed', dayfirst=True)

# Sort
nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)

# Validate nav > 0
invalid = nav[nav['nav'] <= 0]
if not invalid.empty:
    print(f"⚠️ Dropping {len(invalid)} rows with NAV <= 0")
    nav = nav[nav['nav'] > 0]

# Reindex to business days and forward-fill
def reindex_fund(group):
    amfi = group['amfi_code'].iloc[0] if 'amfi_code' in group.columns else group.name
    group = group.set_index('date')
    group = group[~group.index.duplicated(keep='last')]
    if group.empty:
        return group
    idx = pd.bdate_range(start=group.index.min(), end=group.index.max())
    group = group.reindex(idx)
    group['amfi_code'] = amfi
    group['nav'] = group['nav'].ffill()
    return group.rename_axis('date').reset_index()

nav_clean = nav.groupby('amfi_code', group_keys=False).apply(reindex_fund).dropna(subset=['nav'])
nav_clean = nav_clean.sort_values(['amfi_code', 'date']).reset_index(drop=True)
nav_clean = nav_clean[['amfi_code', 'date', 'nav']]

nav_clean.to_csv(PROCESSED_DIR / "clean_nav.csv", index=False)
print(f"\n✅ Cleaned NAV: {len(nav_raw):,} → {len(nav_clean):,} rows")
print(f"   Forward-filled {len(nav_clean) - len(nav):,} weekend/holiday gaps")
display(nav_clean.head(10))

Raw NAV records: 46,000
Duplicates     : 0
Null NAVs      : 0


C:\Users\Dell\AppData\Local\Temp\ipykernel_20664\2192134781.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nav_clean = nav.groupby('amfi_code', group_keys=False).apply(reindex_fund).dropna(subset=['nav'])



✅ Cleaned NAV: 46,000 → 46,000 rows
   Forward-filled 0 weekend/holiday gaps


,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
5,100016,2022-01-10,510.7136
6,100016,2022-01-11,513.5542
7,100016,2022-01-12,512.3195
8,100016,2022-01-13,510.2445
9,100016,2022-01-14,514.3636


## 2. Clean Investor Transactions

- Parse `transaction_date`
- Standardize `transaction_type` to `SIP`, `Lumpsum`, `Redemption`
- Validate `amount_inr > 0` and `kyc_status ∈ {Verified, Pending}`

In [3]:
tx_raw = pd.read_csv(RAW_DIR / "08_investor_transactions.csv")
print(f"Raw transactions: {len(tx_raw):,}")
print(f"Transaction types: {tx_raw['transaction_type'].unique()}")

tx = tx_raw.copy()

# Parse dates
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'], format='mixed', dayfirst=True)

# Standardize transaction_type
tx['transaction_type'] = tx['transaction_type'].str.strip().str.title()
type_map = {'Sip': 'SIP', 'Lumpsum': 'Lumpsum', 'Redemption': 'Redemption'}
tx['transaction_type'] = tx['transaction_type'].map(type_map).fillna(tx['transaction_type'])
tx = tx[tx['transaction_type'].isin(['SIP', 'Lumpsum', 'Redemption'])]

# Validate amount > 0
invalid_amt = tx[tx['amount_inr'] <= 0]
if not invalid_amt.empty:
    print(f"⚠️ Dropping {len(invalid_amt)} rows with amount <= 0")
    tx = tx[tx['amount_inr'] > 0]

# Validate KYC
tx['kyc_status'] = tx['kyc_status'].str.strip().str.title()
tx = tx[tx['kyc_status'].isin(['Verified', 'Pending'])]

tx.to_csv(PROCESSED_DIR / "clean_transactions.csv", index=False)
print(f"\n✅ Cleaned transactions: {len(tx_raw):,} → {len(tx):,} rows")
print(f"\nTransaction type distribution:")
display(tx['transaction_type'].value_counts())
print(f"\nKYC status distribution:")
display(tx['kyc_status'].value_counts())

Raw transactions: 32,778
Transaction types: ['SIP' 'Redemption' 'Lumpsum']



✅ Cleaned transactions: 32,778 → 32,778 rows

Transaction type distribution:


transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64


KYC status distribution:


kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

## 3. Clean Scheme Performance

- Coerce return/ratio columns to numeric
- Flag negative Sharpe ratios
- Validate expense ratio range (0.1%–2.5%)

In [4]:
perf_raw = pd.read_csv(RAW_DIR / "07_scheme_performance.csv")
print(f"Raw performance records: {len(perf_raw)}")

perf = perf_raw.copy()

numeric_cols = [
    'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
    'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio',
    'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct'
]

for col in numeric_cols:
    if col in perf.columns:
        perf[col] = pd.to_numeric(perf[col], errors='coerce')

# Flag negative Sharpe
neg_sharpe = perf[perf['sharpe_ratio'] < 0]
if not neg_sharpe.empty:
    print(f"⚠️ {len(neg_sharpe)} schemes with negative Sharpe ratio")

# Validate expense ratio
perf['expense_ratio_pct'] = pd.to_numeric(perf['expense_ratio_pct'], errors='coerce')
anomalies = perf[(perf['expense_ratio_pct'] < 0.1) | (perf['expense_ratio_pct'] > 2.5)]
if not anomalies.empty:
    print(f"⚠️ {len(anomalies)} schemes with expense_ratio outside 0.1%–2.5%")

perf.to_csv(PROCESSED_DIR / "clean_performance.csv", index=False)
print(f"\n✅ Cleaned performance: {len(perf)} rows")
display(perf[['scheme_name', 'sharpe_ratio', 'sortino_ratio', 'return_3yr_pct', 'risk_grade']].head(10))

Raw performance records: 40

✅ Cleaned performance: 40 rows


,scheme_name,sharpe_ratio,sortino_ratio,return_3yr_pct,risk_grade
0,SBI Bluechip Fund - Regular Plan - Growth,0.88,1.29,12.36,Moderate
1,SBI Bluechip Fund - Direct Plan - Growth,0.81,1.29,11.30,Moderate
2,SBI Small Cap Fund - Regular Plan - Growth,0.94,1.35,23.39,Very High
3,SBI Small Cap Fund - Direct Plan - Growth,0.93,1.67,23.14,Very High
4,SBI Magnum Gilt Fund - Regular Plan - Growth,1.52,2.11,6.07,Low
5,HDFC Top 100 Fund - Regular Plan - Growth,1.06,1.70,14.84,Moderate
6,HDFC Top 100 Fund - Direct Plan - Growth,0.96,1.45,13.38,Moderate
7,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,0.87,1.44,16.58,High
8,HDFC Mid-Cap Opportunities Fund - Direct - Growth,0.80,1.38,15.29,High
9,HDFC Short Term Debt Fund - Regular - Growth,1.84,2.79,7.37,Low


## 4. Verify SQLite Database Tables

Connect to `bluestock_mf.db` and verify all expected tables are loaded with correct row counts.

In [5]:
if DB_PATH.exists():
    conn = sqlite3.connect(str(DB_PATH))
    cursor = conn.cursor()

    # List all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    tables = [row[0] for row in cursor.fetchall()]
    print(f"Tables in bluestock_mf.db ({len(tables)}):")

    table_info = []
    for tbl in tables:
        cursor.execute(f"SELECT COUNT(*) FROM [{tbl}]")
        count = cursor.fetchone()[0]
        table_info.append({"Table": tbl, "Rows": f"{count:,}"})
        print(f"  📋 {tbl:30s} {count:>8,} rows")

    display(pd.DataFrame(table_info))
    conn.close()
else:
    print("⚠️ Database not found. Run `python scripts/etl_pipeline.py` first.")

Tables in bluestock_mf.db (12):
  📋 benchmark_indices                 8,050 rows
  📋 category_inflows                    144 rows
  📋 dim_date                          1,797 rows
  📋 dim_fund                             40 rows
  📋 fact_aum                             90 rows
  📋 fact_nav                         51,400 rows
  📋 fact_performance                     40 rows
  📋 fact_portfolio                      322 rows
  📋 fact_sip_industry                    48 rows
  📋 fact_transactions                32,778 rows
  📋 industry_folio_count                 21 rows
  📋 sqlite_sequence                       7 rows


,Table,Rows
0,benchmark_indices,"8,050"
1,category_inflows,144
2,dim_date,"1,797"
3,dim_fund,40
4,fact_aum,90
5,fact_nav,"51,400"
6,fact_performance,40
7,fact_portfolio,322
8,fact_sip_industry,48
9,fact_transactions,"32,778"


## 5. Execute Analytical Queries from `queries.sql`

Run the 10 pre-defined queries and display results inline.

In [6]:
import re

if DB_PATH.exists():
    conn = sqlite3.connect(str(DB_PATH))
    queries_file = SQL_DIR / "queries.sql"

    if queries_file.exists():
        sql_text = queries_file.read_text(encoding="utf-8")

        # Split on comment lines that start with '--' and contain a query title
        # Each query block = comment header + SQL
        blocks = re.split(r'(?=^-- \d+\.)', sql_text, flags=re.MULTILINE)
        blocks = [b.strip() for b in blocks if b.strip()]

        for i, block in enumerate(blocks, 1):
            lines = block.strip().split('\n')
            # Extract title from first comment line
            title = lines[0].lstrip('- ').strip() if lines[0].startswith('--') else f"Query {i}"
            # Extract SQL (non-comment lines)
            sql = '\n'.join(l for l in lines if not l.strip().startswith('--')).strip()
            if not sql:
                continue

            print(f"\n{'=' * 70}")
            print(f"📊 {title}")
            print(f"{'=' * 70}")
            try:
                result = pd.read_sql_query(sql, conn)
                display(result.head(10))
                print(f"   ({len(result)} rows returned)")
            except Exception as e:
                print(f"   ❌ Error: {e}")
    else:
        print("⚠️ queries.sql not found")

    conn.close()
else:
    print("⚠️ Database not found. Run `python scripts/etl_pipeline.py` first.")


📊 1. Top 5 funds by AUM (from fact_performance, order by aum_crore DESC)


,scheme_name,fund_house,aum_crore
0,Mirae Asset Emerging Bluechip Fund - Regular -...,Mirae Asset MF,49046.0
1,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,47469.0
2,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,43630.0
3,DSP Top 100 Equity Fund - Regular - Growth,DSP Mutual Fund,41828.0
4,UTI Mid Cap Fund - Regular - Growth,UTI Mutual Fund,41728.0


   (5 rows returned)

📊 2. Average NAV per month across all funds (from fact_nav, group by year+month)


,year_month,avg_nav
0,2022-01,209.919982
1,2022-02,210.148631
2,2022-03,210.905818
3,2022-04,213.250844
4,2022-05,214.023917
5,2022-06,213.508315
6,2022-07,214.235181
7,2022-08,215.865051
8,2022-09,217.585336
9,2022-10,218.216087


   (60 rows returned)

📊 3. SIP inflow year-on-year growth (from fact_sip_industry, compare year totals)


...

   (10 rows returned)

📊 9. Funds with Sharpe ratio > 1 (from fact_performance)


,scheme_name,fund_house,sharpe_ratio
0,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,7.68
1,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,6.18
2,ABSL Liquid Fund - Regular - Growth,Aditya Birla Sun Life MF,5.14
3,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,1.84
4,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,1.52
5,Nippon India Gilt Securities Fund - Regular - ...,Nippon India MF,1.33
6,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,1.06
7,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset MF,1.06
8,ICICI Pru Bluechip Fund - Direct - Growth,ICICI Prudential MF,1.03


   (9 rows returned)

📊 10. Monthly transaction volume count (from fact_transactions, group by year+month)


,year_month,transaction_volume_count
0,2024-01,1959
1,2024-02,1822
2,2024-03,2011
3,2024-04,1922
4,2024-05,1934
5,2024-06,1951
6,2024-07,1957
7,2024-08,1976
8,2024-09,1871
9,2024-10,1974


   (24 rows returned)


---
*Day 2 Data Cleaning & Database Loading complete. All 3 core datasets cleaned, SQLite DB verified, and analytical queries executed.*